In [1]:
from pyspark.sql.functions import *

# --------------------------------------------
# Load tables
# --------------------------------------------

silver_df = spark.table("lh_Silver_Telecom.dbo.silver_network_events")

dim_date = spark.table("lh_Gold_Telecom.dbo.dim_date")
dim_site = spark.table("lh_Gold_Telecom.dbo.dim_site").filter(col("is_current") == True)
dim_vendor = spark.table("lh_Gold_Telecom.dbo.dim_vendor")
dim_technology = spark.table("lh_Gold_Telecom.dbo.dim_technology")
dim_province = spark.table("lh_Gold_Telecom.dbo.dim_province")
dim_cause = spark.table("lh_Gold_Telecom.dbo.dim_cause")


StatementMeta(, 123d9369-010f-4546-9e8b-8fbff8e04f8f, 3, Finished, Available, Finished, False)

In [2]:
# --------------------------------------------
# Create temp views 
# --------------------------------------------

silver_df.createOrReplaceTempView("silver")
dim_date.createOrReplaceTempView("dim_date")
dim_site.createOrReplaceTempView("dim_site")
dim_vendor.createOrReplaceTempView("dim_vendor")
dim_technology.createOrReplaceTempView("dim_technology")
dim_province.createOrReplaceTempView("dim_province")
dim_cause.createOrReplaceTempView("dim_cause")


StatementMeta(, 123d9369-010f-4546-9e8b-8fbff8e04f8f, 4, Finished, Available, Finished, False)

In [3]:
%%sql
CREATE OR REPLACE TEMP VIEW fact_src AS
SELECT
    d.date_key,
    s.site_id,
    v.vendor_key,
    t.technology_key,
    p.province_key,
    c.cause_key,

    silver.duration_minutes_corrected,
    silver.availability_percent_corrected,
    silver.is_sla_breach_indicator

FROM silver
JOIN dim_date d
    ON silver.outage_start_date = d.calendar_date
JOIN dim_site s
    ON silver.Site_ID = s.site_id
   AND s.is_current = true
JOIN dim_vendor v
    ON silver.Vendor = v.vendor_name
JOIN dim_technology t
    ON silver.Technology = t.technology_name
JOIN dim_province p
    ON silver.Province_Code = p.province_code
JOIN dim_cause c
    ON silver.Cause = c.cause_name;


StatementMeta(, 123d9369-010f-4546-9e8b-8fbff8e04f8f, 5, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [4]:
%%sql
SELECT COUNT(*) FROM fact_src;


StatementMeta(, 123d9369-010f-4546-9e8b-8fbff8e04f8f, 6, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [7]:
%%sql
CREATE OR REPLACE TEMP VIEW fact_network_availability_agg AS
SELECT
    date_key,
    site_id,
    vendor_key,
    technology_key,
    province_key,
  

    SUM(duration_minutes_corrected)       AS total_outage_minutes,
    COUNT(*)                              AS event_count,
    AVG(availability_percent_corrected)   AS sla_compliance_pct,
    MAX(is_sla_breach_indicator)           AS sla_breach_flag

FROM fact_src
GROUP BY
    date_key,
    site_id,
    vendor_key,
    technology_key,
    province_key


StatementMeta(, dd65c641-afc9-4ef8-ac27-fa3d5ab72ce1, 9, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [6]:
%%sql
CREATE OR REPLACE TEMP VIEW fact_network_availability_agg AS
SELECT
    date_key,
    site_id,
    vendor_key,
    technology_key,
    province_key,
  

   SUM(duration_minutes_corrected) AS total_outage_minutes,
    COUNT(*) AS event_count,
    
    -- Calculate availability from TOTAL downtime
    ((1440 - SUM(duration_minutes_corrected)) / 1440.0) * 100 AS sla_compliance_pct,
    
    MAX(is_sla_breach_indicator) AS sla_breach_flag

FROM fact_src
GROUP BY
    date_key,
    site_id,
    vendor_key,
    technology_key,
    province_key


StatementMeta(, 123d9369-010f-4546-9e8b-8fbff8e04f8f, 8, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [7]:
%%sql
CREATE OR REPLACE TABLE lh_Gold_Telecom.dbo.fact_network_availability
USING DELTA
AS
SELECT * FROM fact_network_availability_agg;


StatementMeta(, 123d9369-010f-4546-9e8b-8fbff8e04f8f, 9, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [8]:
# Verify table was created
print("✅ Gold tables created successfully!\n")
print(f"fact_network_availability rows: {spark.table('lh_Gold_Telecom.dbo.fact_network_availability').count()}")

StatementMeta(, 123d9369-010f-4546-9e8b-8fbff8e04f8f, 10, Finished, Available, Finished, False)

✅ Gold tables created successfully!

fact_network_availability rows: 58295


In [9]:
df = spark.sql("SELECT * FROM lh_Gold_Telecom.dbo.fact_network_availability LIMIT 1000")
display(df)

StatementMeta(, 123d9369-010f-4546-9e8b-8fbff8e04f8f, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, af8f0f9e-f39b-4cc6-836d-faed9e1fd1bf)